# M3L4 E02 — Tracing de un sistema multiagente mock
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

**Objetivo:** aplicar el MiniTracer a un sistema multiagente real con orquestador y agentes especialistas.

## Arquitectura
```
Usuario
  ↓
Orchestrator  (routing)
  ├── HRAgent
  ├── ITAgent
  ├── FinanceAgent
  └── LegalAgent
  ↓
Respuesta final
```

## Paso 1 — Copiar el MiniTracer del E01

> Copiamos la clase completa para que el notebook sea autocontenido.

In [ ]:
import time
import uuid
from datetime import datetime

class MiniTracer:
    def __init__(self):
        self.traces = []

    def start_trace(self, name, input_data=None, metadata=None, tags=None):
        trace = {
            'trace_id': str(uuid.uuid4()),
            'name': name,
            'input': input_data,
            'output': None,
            'metadata': metadata or {},
            'tags': tags or [],
            'spans': [],
            'created_at': datetime.utcnow().isoformat(),
            '_started_at_ts': time.time()
        }
        self.traces.append(trace)
        return trace

    def add_span(self, trace, name, input_data=None, output_data=None, metadata=None, duration_ms=None):
        span = {
            'span_id': str(uuid.uuid4()),
            'name': name,
            'input': input_data,
            'output': output_data,
            'metadata': metadata or {},
            'duration_ms': duration_ms
        }
        trace['spans'].append(span)
        return span

    def update_trace_output(self, trace, output_data):
        trace['output'] = output_data
        trace['total_duration_ms'] = round((time.time() - trace['_started_at_ts']) * 1000, 2)

    def show_trace(self, trace):
        return trace

print('MiniTracer listo.')

## Paso 2 — Sistema multiagente (ya dado)

El router y los agentes ya están implementados. Tu tarea es **instrumentarlos con trazas**.

In [ ]:
def route_query(query: str) -> str:
    q = query.lower()
    if any(w in q for w in ['vacaciones', 'licencia', 'recibo', 'nómina', 'portal rrhh']):
        return 'hr'
    if any(w in q for w in ['vpn', 'laptop', 'error', 'app', 'wifi', 'login', 'contraseña']):
        return 'it'
    if any(w in q for w in ['factura', 'pago', 'reembolso', 'gasto', 'cobro']):
        return 'finance'
    if any(w in q for w in ['contrato', 'legal', 'confidencialidad', 'nda']):
        return 'legal'
    if len(q.split()) <= 2:
        return 'clarification'
    return 'general'

def hr_agent(query: str) -> str:
    return 'Soy HRAgent. Te ayudo con vacaciones, licencias, recibos y temas de RRHH.'

def it_agent(query: str) -> str:
    return 'Soy ITAgent. Te ayudo con VPN, laptop, accesos y errores técnicos.'

def finance_agent(query: str) -> str:
    return 'Soy FinanceAgent. Te ayudo con facturas, pagos y reembolsos.'

def legal_agent(query: str) -> str:
    return 'Soy LegalAgent. Te ayudo con contratos y temas legales.'

def general_agent(query: str) -> str:
    return 'Necesito más contexto para ayudarte correctamente.'

agents = {
    'hr': hr_agent,
    'it': it_agent,
    'finance': finance_agent,
    'legal': legal_agent,
    'general': general_agent,
    'clarification': general_agent
}

print('Agentes listos.')

## Paso 3 — TODO: Instrumentar con trazas

Completá la función `handle_query_with_tracing` para que:
1. Cree un trace al inicio con el query
2. Agregue un span para el routing (con intent detectado y duración real)
3. Agregue un span para el agente seleccionado (con respuesta y duración real)
4. Actualice el output del trace al final

In [ ]:
def handle_query_with_tracing(query: str, tracer: MiniTracer) -> dict:
    """
    Procesa una query con un sistema multiagente e instrumenta cada paso con trazas.

    Args:
        query: consulta del usuario
        tracer: instancia de MiniTracer

    Returns:
        trace: el trace completo con todos los spans
    """
    # TODO 1: crear el trace con nombre 'multiagent-support-request'
    # metadata: {'environment': 'notebook', 'router_version': 'v1'}
    # tags: ['multiagent', 'm3l4']
    trace = None  # reemplazar

    # TODO 2: medir el tiempo real del routing y agregar span 'orchestrator-routing'
    # input: {'query': query}
    # output: {'intent': intent}
    # metadata: {'router_version': 'v1'}
    intent = None  # reemplazar con route_query(query)

    # TODO 3: medir el tiempo real del agente y agregar span '{intent}-agent'
    # input: {'query': query}
    # output: {'response': response}
    response = None  # reemplazar con agents.get(intent, general_agent)(query)

    # TODO 4: actualizar el output del trace
    # {'intent': intent, 'final_response': response}

    return trace

print('Función definida.')

In [ ]:
tracer = MiniTracer()

trace = handle_query_with_tracing('No puedo ver mi factura', tracer)
trace

## Paso 4 — Ejecutar múltiples consultas

In [ ]:
queries = [
    '¿Cómo solicito mis días de vacaciones?',
    'Mi VPN no conecta desde ayer',
    'Necesito ver mi factura del mes pasado',
    'Necesito el contrato de confidencialidad actualizado',
    'ayuda'
]

tracer2 = MiniTracer()
for q in queries:
    t = handle_query_with_tracing(q, tracer2)
    if t:
        print(f"Query: {q[:45]}...")
        print(f"  Intent: {t['output']['intent'] if t['output'] else 'N/A'}")
        print(f"  Spans: {[s['name'] for s in t['spans']]}")
        print()

## Checks automáticos

In [ ]:
tracer_check = MiniTracer()

t1 = handle_query_with_tracing('No puedo ver mi factura', tracer_check)
assert t1 is not None, 'trace es None'
assert len(t1['spans']) == 2, f'Se esperaban 2 spans, hay {len(t1["spans"])}'
assert t1['spans'][0]['name'] == 'orchestrator-routing', 'Primer span incorrecto'
assert t1['output']['intent'] == 'finance', f"Intent incorrecto: {t1['output']['intent']}"
assert t1['spans'][0]['output']['intent'] == 'finance'
assert t1['spans'][1]['output']['response'] is not None
assert t1['spans'][0]['duration_ms'] is not None
assert t1['spans'][1]['duration_ms'] is not None

print('Checks E02 OK ✅')